In [1]:
import pandas as pd
import numpy as np

def engineer_features(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()

    # ── 1. Total square footage ───────────────────────────
    df['TotalSF'] = (df['TotalBsmtSF'] + df['1stFlrSF'] + df['2ndFlrSF'])

    # ── 2. Total bathrooms ────────────────────────────────
    df['TotalBaths'] = (df['FullBath'] + 0.5 * df['HalfBath'] + df['BsmtFullBath'] + 0.5 * df['BsmtHalfBath'])

    # ── 3. Age of house at time of sale ───────────────────
    df['HouseAge']  = df['YrSold'] - df['YearBuilt']
    df['RemodAge']  = df['YrSold'] - df['YearRemodAdd']

    # ── 4. Has features (binary indicators) ───────────────
    df['HasGarage'] = (df['GarageArea'] > 0).astype(int)
    df['HasPool']   = (df['PoolArea']   > 0).astype(int)
    df['HasBsmt']   = (df['TotalBsmtSF']> 0).astype(int)

    # ── 5. Quality × Area interaction ────────────────────
    # High quality + large area = premium pricing
    df['QualArea']  = df['OverallQual'] * df['GrLivArea']

    # ── 6. Drop originals that are now redundant ──────────
    df.drop(columns=['1stFlrSF', '2ndFlrSF','FullBath', 'HalfBath'], inplace=True)

    return df

In [11]:
from pathlib import Path
import sys
sys.path.append('../')
from src.data_loader import data_loader
from src.preprocess import handle_missing

df = data_loader()

df = handle_missing(df)

df = engineer_features(df)

df

,Id,MSSubClass,MSZoning,LotFrontage,LotArea,Street,Alley,LotShape,LandContour,Utilities,...,SaleType,SaleCondition,TotalSF,TotalBaths,HouseAge,RemodAge,HasGarage,HasPool,HasBsmt,QualArea
0,1461,20,RH,80.0,11622,Pave,None,Reg,Lvl,AllPub,...,WD,Normal,1778.0,1.0,49,49,1,0,1,4480
1,1462,20,RL,81.0,14267,Pave,None,IR1,Lvl,AllPub,...,WD,Normal,2658.0,1.5,52,52,1,0,1,7974
2,1463,60,RL,74.0,13830,Pave,None,IR1,Lvl,AllPub,...,WD,Normal,2557.0,2.5,13,12,1,0,1,8145
3,1464,60,RL,78.0,9978,Pave,None,IR1,Lvl,AllPub,...,WD,Normal,2530.0,2.5,12,12,1,0,1,9624
4,1465,120,RL,43.0,5005,Pave,None,IR1,HLS,AllPub,...,WD,Normal,2560.0,2.0,18,18,1,0,1,10240
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1454,2915,160,RM,21.0,1936,Pave,None,Reg,Lvl,AllPub,...,WD,Normal,1638.0,1.5,36,36,0,0,1,4368
1455,2916,160,RM,21.0,1894,Pave,None,Reg,Lvl,AllPub,...,WD,Abnorml,1638.0,1.5,36,36,1,0,1,4368
1456,2917,20,RL,160.0,20000,Pave,None,Reg,Lvl,AllPub,...,WD,Abnorml,2448.0,2.0,46,10,1,0,1,6120
1457,2918,85,RL,62.0,10441,Pave,None,Reg,Lvl,AllPub,...,WD,Normal,1882.0,1.5,14,14,0,0,1,4850


In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import (StandardScaler, OrdinalEncoder, PowerTransformer)
from sklearn.impute import SimpleImputer
from sklearn.linear_model import Ridge, Lasso, ElasticNet
from sklearn.model_selection import cross_val_score


def build_pipeline(model, numeric_cols, categorical_cols):
    # Numeric: impute remaining NaNs → scale
    numeric_transformer = Pipeline([
        ('imputer', SimpleImputer(strategy='median')),
        ('scaler',  StandardScaler()),
    ])

    # Categorical: impute → encode
    categorical_transformer = Pipeline([
        ('imputer', SimpleImputer(strategy='most_frequent')),
        ('encoder', OrdinalEncoder(
            handle_unknown='use_encoded_value',
            unknown_value=-1)),
    ])

    # Combine both
    preprocessor = ColumnTransformer([
        ('num', numeric_transformer,  numeric_cols),
        ('cat', categorical_transformer, categorical_cols),
    ])

    # Full pipeline: preprocess + model
    return Pipeline([
        ('preprocessor', preprocessor),
        ('model', model),
    ])

def train_and_cv(pipeline, X_train, y_train, cv=5):
    """Returns mean RMSE on log-transformed target."""
    scores = cross_val_score(
        pipeline, X_train, y_train,
        scoring='neg_root_mean_squared_error',
        cv=cv, n_jobs=-1
    )
    return -scores.mean(), scores.std()

